# 🔬 Unsupervised Learning: DBSCAN & GMM
**Dataset:** `clustering_dataset.csv` (530 samples, 2 features — 5 blobs + noise)

This notebook covers:
1. EDA & data preparation
2. DBSCAN — with hyperparameter tuning
3. GMM — with hyperparameter tuning
4. Evaluation & comparison

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
                     'axes.grid': True, 'grid.alpha': 0.3})
print('Libraries loaded ✓')

## 1. Load & Explore Dataset

In [ ]:
df = pd.read_csv('../data/clustering_dataset.csv')
print('Shape:', df.shape)
df.head(10)

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Scatter
axes[0].scatter(df['feature_1'], df['feature_2'], alpha=0.5, s=15, color='steelblue')
axes[0].set_title('Raw Data Scatter')
axes[0].set_xlabel('feature_1'); axes[0].set_ylabel('feature_2')

# Histograms
for i, col in enumerate(['feature_1', 'feature_2']):
    axes[i+1].hist(df[col], bins=35, color=['steelblue','coral'][i], edgecolor='white', alpha=0.8)
    axes[i+1].set_title(f'{col} Distribution')

plt.tight_layout()
plt.show()

## 2. Preprocessing

In [ ]:
X = df[['feature_1', 'feature_2']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Original range  feature_1:', X[:,0].min().round(2), '→', X[:,0].max().round(2))
print('Scaled  range   feature_1:', X_scaled[:,0].min().round(2), '→', X_scaled[:,0].max().round(2))

## 3. DBSCAN
### 3.1 How to Choose ε — k-distance Elbow Plot

In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 5  # same as min_samples we plan to use
nbrs = NearestNeighbors(n_neighbors=k).fit(X_scaled)
distances, _ = nbrs.kneighbors(X_scaled)
k_dist = np.sort(distances[:, k-1])[::-1]

plt.figure(figsize=(8, 4))
plt.plot(k_dist, color='steelblue', linewidth=1.5)
plt.axhline(y=0.5, color='red', linestyle='--', label='eps ≈ 0.5')
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{k}-NN distance')
plt.title('k-Distance Graph (Elbow = optimal eps)')
plt.legend()
plt.tight_layout()
plt.show()

### 3.2 Fit DBSCAN with Chosen Hyperparameters

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
EPS         = 0.5       # neighborhood radius
MIN_SAMPLES = 5         # core point threshold
METRIC      = 'euclidean'
ALGORITHM   = 'auto'
LEAF_SIZE   = 30
# ─────────────────────────────────────────────────────────────────────────────

dbscan = DBSCAN(
    eps=EPS,
    min_samples=MIN_SAMPLES,
    metric=METRIC,
    algorithm=ALGORITHM,
    leaf_size=LEAF_SIZE
)
db_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise    = (db_labels == -1).sum()
print(f'Clusters found : {n_clusters}')
print(f'Noise points   : {n_noise}')

In [ ]:
PALETTE = ['#4c72b0','#dd8452','#55a868','#c44e52','#8172b2',
           '#937860','#da8bc3','#8c8c8c','#ccb974','#64b5cd']

def get_colors(labels):
    cmap = {}
    ci = 0
    for u in sorted(set(labels)):
        cmap[u] = 'red' if u == -1 else PALETTE[ci % len(PALETTE)]
        if u != -1: ci += 1
    return [cmap[l] for l in labels]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, X_p, title in zip(axes,
    [X, X_scaled], ['Original Space', 'Scaled Space']):
    ax.scatter(X_p[:,0], X_p[:,1], c=get_colors(db_labels), s=20, alpha=0.8)
    ax.set_title(f'DBSCAN — {title}  (eps={EPS}, min_samples={MIN_SAMPLES})')
    ax.set_xlabel('feature_1'); ax.set_ylabel('feature_2')
    # legend
    from matplotlib.lines import Line2D
    handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor='red',
                      markersize=8, label='Noise (-1)')]
    for c in range(n_clusters):
        handles.append(Line2D([0],[0], marker='o', color='w',
                               markerfacecolor=PALETTE[c % len(PALETTE)],
                               markersize=8, label=f'Cluster {c}'))
    ax.legend(handles=handles, fontsize=8, loc='best')
plt.tight_layout()
plt.show()

### 3.3 DBSCAN Hyperparameter Sweep

In [ ]:
eps_vals = np.arange(0.1, 2.1, 0.2)
ms_vals  = [2, 4, 6, 8, 10]
results  = []

for ep in eps_vals:
    for ms in ms_vals:
        lbl  = DBSCAN(eps=ep, min_samples=ms).fit_predict(X_scaled)
        mask = lbl != -1
        ncl  = len(set(lbl[mask]))
        sil  = silhouette_score(X_scaled[mask], lbl[mask]) if ncl >= 2 and mask.sum() > ncl else np.nan
        results.append({'eps': round(ep,2), 'min_samples': ms,
                         'n_clusters': ncl, 'n_noise': (lbl==-1).sum(),
                         'silhouette': round(sil, 4) if not np.isnan(sil) else np.nan})

res_df = pd.DataFrame(results)
pivot  = res_df.pivot(index='min_samples', columns='eps', values='silhouette')

plt.figure(figsize=(12, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Silhouette Score'})
plt.title('DBSCAN — Silhouette Score Heatmap (eps × min_samples)')
plt.tight_layout()
plt.show()

res_df.sort_values('silhouette', ascending=False).head(10)

## 4. GMM
### 4.1 Choose n_components via BIC / AIC

In [ ]:
comp_range = range(2, 13)
bics, aics = [], []
for nc in comp_range:
    gm = GaussianMixture(n_components=nc, covariance_type='full',
                         random_state=42, max_iter=200)
    gm.fit(X_scaled)
    bics.append(gm.bic(X_scaled))
    aics.append(gm.aic(X_scaled))

plt.figure(figsize=(8, 4))
plt.plot(list(comp_range), bics, 'o-', label='BIC', color='steelblue', linewidth=2)
plt.plot(list(comp_range), aics, 's-', label='AIC', color='coral',    linewidth=2)
best_bic = list(comp_range)[np.argmin(bics)]
plt.axvline(best_bic, color='green', linestyle='--', label=f'Best BIC @ n={best_bic}')
plt.xlabel('n_components'); plt.ylabel('Information Criterion')
plt.title('GMM Model Selection — BIC and AIC')
plt.legend(); plt.tight_layout(); plt.show()
print(f'Best n_components by BIC: {best_bic}')

### 4.2 Fit GMM with Chosen Hyperparameters

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
N_COMPONENTS    = 5
COVARIANCE_TYPE = 'full'       # 'full' | 'tied' | 'diag' | 'spherical'
MAX_ITER        = 200
N_INIT          = 5
INIT_PARAMS     = 'kmeans'     # 'kmeans' | 'k-means++' | 'random'
REG_COVAR       = 1e-6
TOL             = 1e-4
# ─────────────────────────────────────────────────────────────────────────────

gmm = GaussianMixture(
    n_components=N_COMPONENTS,
    covariance_type=COVARIANCE_TYPE,
    max_iter=MAX_ITER,
    n_init=N_INIT,
    init_params=INIT_PARAMS,
    reg_covar=REG_COVAR,
    tol=TOL,
    random_state=42
)
gmm.fit(X_scaled)
gmm_labels = gmm.predict(X_scaled)
gmm_proba  = gmm.predict_proba(X_scaled)

print('Converged:', gmm.converged_)
print('n_iter   :', gmm.n_iter_)
print('Log-likelihood (lower_bound):', round(gmm.lower_bound_, 4))

In [ ]:
from matplotlib.patches import Ellipse

def draw_ellipse(ax, mean, cov, color, n_std=2.0, alpha=0.25):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(mean, w, h, angle=theta,
                  facecolor=color, alpha=alpha, edgecolor=color, linewidth=2)
    ax.add_patch(ell)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter
ax = axes[0]
ax.scatter(X_scaled[:,0], X_scaled[:,1], c=[PALETTE[l % len(PALETTE)] for l in gmm_labels],
           s=20, alpha=0.7)
for i, (mean, cov) in enumerate(zip(gmm.means_, gmm.covariances_)):
    draw_ellipse(ax, mean, cov, PALETTE[i % len(PALETTE)])
ax.set_title('GMM Clusters with Covariance Ellipses')
ax.set_xlabel('feature_1 (scaled)'); ax.set_ylabel('feature_2 (scaled)')

# Probability density
ax2 = axes[1]
x_min, x_max = X_scaled[:,0].min()-1, X_scaled[:,0].max()+1
y_min, y_max = X_scaled[:,1].min()-1, X_scaled[:,1].max()+1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                     np.linspace(y_min, y_max, 200))
Z = -gmm.score_samples(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
cf = ax2.contourf(xx, yy, Z, levels=30, cmap='plasma', alpha=0.7)
ax2.scatter(X_scaled[:,0], X_scaled[:,1], c='white', s=8, alpha=0.4)
plt.colorbar(cf, ax=ax2, label='−log likelihood')
ax2.set_title('GMM Probability Density')

plt.tight_layout()
plt.show()

### 4.3 Soft Cluster Probabilities

In [ ]:
# Visualise max probability (confidence)
max_proba = gmm_proba.max(axis=1)

plt.figure(figsize=(7, 5))
sc = plt.scatter(X_scaled[:,0], X_scaled[:,1], c=max_proba, cmap='RdYlGn', s=20, alpha=0.9)
plt.colorbar(sc, label='Max cluster probability')
plt.title('GMM — Cluster Assignment Confidence')
plt.xlabel('feature_1 (scaled)'); plt.ylabel('feature_2 (scaled)')
plt.tight_layout(); plt.show()

## 5. Evaluation & Comparison

In [ ]:
def evaluate(name, X, labels):
    mask = labels != -1
    Xm, lm = X[mask], labels[mask]
    nc = len(set(lm))
    if nc >= 2 and len(Xm) > nc:
        sil = silhouette_score(Xm, lm)
        db  = davies_bouldin_score(Xm, lm)
        ch  = calinski_harabasz_score(Xm, lm)
    else:
        sil = db = ch = float('nan')
    return {'Model': name, 'n_clusters': nc, 'n_noise': (labels==-1).sum(),
            'Silhouette ↑': round(sil, 4), 'Davies-Bouldin ↓': round(db, 4),
            'Calinski-Harabasz ↑': round(ch, 2)}

eval_df = pd.DataFrame([
    evaluate('DBSCAN', X_scaled, db_labels),
    evaluate('GMM',    X_scaled, gmm_labels),
])
eval_df.set_index('Model', inplace=True)
eval_df.style.background_gradient(cmap='Greens', subset=['Silhouette ↑', 'Calinski-Harabasz ↑']) \
             .background_gradient(cmap='Reds_r', subset=['Davies-Bouldin ↓'])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (lbl, title, badge) in zip(axes, [
    (db_labels, f'DBSCAN  (eps={EPS}, min_samples={MIN_SAMPLES})', 'DBSCAN'),
    (gmm_labels, f'GMM  (n={N_COMPONENTS}, cov={COVARIANCE_TYPE})', 'GMM'),
]):
    ax.scatter(X[:,0], X[:,1], c=[PALETTE[l % len(PALETTE)] if l != -1 else 'red' for l in lbl],
               s=18, alpha=0.8)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('feature_1'); ax.set_ylabel('feature_2')

plt.suptitle('Side-by-Side Comparison — Original Feature Space', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Summary

| | DBSCAN | GMM |
|---|---|---|
| **Type** | Density-based | Probabilistic generative |
| **Key params** | eps, min_samples | n_components, covariance_type |
| **Noise handling** | Built-in (-1 label) | No native noise |
| **Cluster shape** | Arbitrary | Elliptical (Gaussian) |
| **Soft assignments** | No | Yes (probabilities) |
| **Model selection** | k-distance elbow | BIC / AIC |
| **Strength** | Robust to outliers | Flexible covariance structures |